In [12]:
import numpy as np
np.random.seed(42)
X=np.array([
    [1,2],
    [2,1],
    [3,4],
    [4,3]
])
y_label=np.array([0,1,2,1])
Y=np.zeros((4,3))
Y[np.arange(4),y_label]=1
layers=[2,16,16,8,3]
lr=0.1
epochs=2000
def relu(z):
    return np.maximum(0,z)
def back_relu(dA,Z):
    dZ=dA.copy()
    dZ[Z<=0]=0
    return dZ
def softmax(z):
    exp_z=np.exp(z-np.max(z,axis=1,keepdims=True))
    return exp_z/np.sum(exp_z,axis=1,keepdims=True)
def loss_fn(Y,y_pred,eps=1e-8):
    return -np.mean(np.sum(Y*np.log(y_pred+eps),axis=1))
def creat_model(layers):
    np.random.seed(42)
    params=[]
    for i in range(len(layers)-1):
        fan_in=layers[i]
        fan_out=layers[i+1]
        w=np.random.randn(fan_in,fan_out)*np.sqrt(2/fan_in)
        b=np.zeros((1,fan_out))
        params.append({"w":w,"b":b})
    model={
        "layers":layers,
        "params":params,
        "caches":None
    }
    return model
def forward(model,X):#A_prev,Z
    params=model["params"]
    caches=[]
    A=X
    for i,layer in enumerate(params):
        w=layer["w"]
        b=layer["b"]
        A_prev=A
        Z=A_prev@w+b
        if i < len(params)-1:
            A=relu(Z)
        else:
            A=softmax(Z)
        caches.append({"A_prev":A_prev,"Z":Z})
    model["caches"]=caches
    return A
def backward(model,Y,y_pred):
    params=model["params"]
    caches=model["caches"]
    n=Y.shape[0]
    L=len(params)
    grads=[None]*L
    dA=None
    for i in reversed(range(L)):
        w=params[i]["w"]
        cach=caches[i]
        A_prev=cach["A_prev"]
        Z=cach["Z"]
        if i == L-1:
            dZ=y_pred-Y
        else:
            dZ=back_relu(dA,Z)
        dw=A_prev.T@dZ/n
        db=np.mean(dZ,axis=0,keepdims=True)
        dA=dZ@w.T
        grads[i]={"dw":dw,"db":db}
    return grads
def update(model,grads,lr):
    params=model["params"]
    for i in range(len(params)):
        params[i]["w"]-=lr*grads[i]["dw"]
        params[i]["b"]-=lr*grads[i]["db"]
def train(model,Y,X,epochs,lr):
    for epoch in range(epochs):
        y_pred=forward(model,X)
        loss=loss_fn(Y,y_pred)
        grads=backward(model,Y,y_pred)
        update(model,grads,lr)
        if epoch % 200 == 0 :
            print(f"epoch{epoch},loss{loss:.6f}")
def predict(model,X):
    y_pred=forward(model,X)
    y_pred_label=np.argmax(y_pred,axis=1)
    return y_pred_label
model=creat_model([2,16,16,8,3])
train(model,Y,X,epochs=epochs,lr=lr)
y_pred_label=predict(model,X)
print("預測",y_pred_label)
print("真實",y_label)
        


epoch0,loss3.371488
epoch200,loss0.004040
epoch400,loss0.001346
epoch600,loss0.000746
epoch800,loss0.000500
epoch1000,loss0.000371
epoch1200,loss0.000291
epoch1400,loss0.000238
epoch1600,loss0.000200
epoch1800,loss0.000172
預測 [0 1 2 1]
真實 [0 1 2 1]
